In [2]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/해사데이터마이닝')
print('설정된 데이터 폴더:', DATA_DIR)
required_files = ['channel_purchase.csv', 'gender_preference.csv']

print('[파일 존재 여부 확인]')
for fname in required_files:
    fpath = DATA_DIR / fname
    print(f'{fname}:', 'OK' if fpath.exists() else '없음')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
설정된 데이터 폴더: /content/drive/MyDrive/해사데이터마이닝
[파일 존재 여부 확인]
channel_purchase.csv: OK
gender_preference.csv: OK


In [5]:
file_path = DATA_DIR / 'channel_purchase.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('[기본 정보]')
print(df.info())
ct = pd.crosstab(df['channel'], df['purchase_yn'])

print('[교차표]')
print(ct)
chi2, p, dof, expected = chi2_contingency(ct)
expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)

print('[검정 결과]')
print('chi-square statistic:', round(chi2, 4))
print('p-value:', round(p, 6))
print('degrees of freedom:', dof)

print('[기대도수]')
print(expected_df)
conversion_rate = pd.crosstab(df['channel'], df['purchase_yn'], normalize='index') * 100

print('[채널별 구매 비율(%)]')
print(conversion_rate.round(2))

print('[최종 해석 출력]')
if p < 0.05:
    print('p-value가 0.05보다 작으므로, 유입채널과 구매여부는 독립이 아니며 서로 관련이 있다고 해석할 수 있습니다.')
else:
    print('p-value가 0.05 이상이므로, 유입채널과 구매여부의 관련성을 확인하기 어렵습니다.')

best_channel = conversion_rate[1].idxmax() if 1 in conversion_rate.columns else conversion_rate.iloc[:, -1].idxmax()
best_rate = conversion_rate[1].max() if 1 in conversion_rate.columns else conversion_rate.iloc[:, -1].max()
print(f'구매 비율이 가장 높은 채널은 {best_channel}이며, 구매 비율은 {best_rate:.2f}%입니다.')

[데이터 상위 5행]
  channel  purchase_yn
0    검색광고            1
1    검색광고            1
2    검색광고            1
3    검색광고            1
4    검색광고            1
[기본 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   channel      530 non-null    object
 1   purchase_yn  530 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 8.4+ KB
None
[교차표]
purchase_yn   0    1
channel             
SNS          55   70
검색광고         50   90
이메일          80   45
직접방문         30  110
[검정 결과]
chi-square statistic: 51.716
p-value: 0.0
degrees of freedom: 3
[기대도수]
purchase_yn          0          1
channel                          
SNS          50.707547  74.292453
검색광고         56.792453  83.207547
이메일          50.707547  74.292453
직접방문         56.792453  83.207547
[채널별 구매 비율(%)]
purchase_yn      0      1
channel                  
SNS          44.00  56.00
검색광고       

In [8]:
file_path = DATA_DIR / 'gender_preference.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('[기본 정보]')
print(df.info())
ct = pd.crosstab(df['gender'], df['content_type'])

print('[교차표]')
print(ct)
chi2, p, dof, expected = chi2_contingency(ct)
expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)

print('[검정 결과]')
print('chi-square statistic:', round(chi2, 4))
print('p-value:', round(p, 6))
print('degrees of freedom:', dof)

print('[기대도수]')
print(expected_df)
ratio = pd.crosstab(df['gender'], df['content_type'], normalize='index') * 100
top_pref = ct.idxmax(axis=1)

print('[성별 내 콘텐츠 선호 비율(%)]')
print(ratio.round(2))

print('[성별별 최다 선호 콘텐츠]')
print(top_pref)

print('[최종 해석 출력]')
if p < 0.05:
    print('p-value가 0.05보다 작으므로, 성별에 따라 콘텐츠 선호 분포가 다르다고 해석할 수 있습니다.')
else:
    print('p-value가 0.05 이상이므로, 성별에 따른 콘텐츠 선호 분포 차이를 확인하기 어렵습니다.')

for g in top_pref.index:
    print(f'{g}의 최다 선호 콘텐츠는 {top_pref[g]}입니다.')

[데이터 상위 5행]
  gender content_type
0     남성        데이터분석
1     남성        데이터분석
2     남성        데이터분석
3     남성        데이터분석
4     남성        데이터분석
[기본 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 330 entries, 0 to 329
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   gender        330 non-null    object
 1   content_type  330 non-null    object
dtypes: object(2)
memory usage: 5.3+ KB
None
[교차표]
content_type  데이터분석  디자인  마케팅  프로그래밍
gender                              
남성               50   18   28     54
여성               42   48   55     35
[검정 결과]
chi-square statistic: 24.6478
p-value: 1.8e-05
degrees of freedom: 3
[기대도수]
content_type      데이터분석   디자인        마케팅      프로그래밍
gender                                             
남성            41.818182  30.0  37.727273  40.454545
여성            50.181818  36.0  45.272727  48.545455
[성별 내 콘텐츠 선호 비율(%)]
content_type  데이터분석    디자인    마케팅  프로그래밍
gender                  